# Lab 6.4 &mdash; Assemble an Agent with create_agent

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Bind a model + a tools list into one agent with create_agent
- See that the agent is a runnable CompiledStateGraph
- Inspect its structure: the model and tools nodes it holds

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Assemble an agent](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-04"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
A LangChain (v1) agent is a **model** + a **tools** list bound together by
**`create_agent(model, tools)`** (from `langchain.agents`). It returns a runnable **`CompiledStateGraph`**
&mdash; a small graph with a **`model`** node (decide) and a **`tools`** node (act) that loop until the
model answers. Building it is deterministic and needs no LLM call; **running** it is the optional cell.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic."""
    return "(computed)"

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    return "(fact)"

print("two tools ready:", calculator.name, "&", lookup.name)

## Your Turn
Assemble the agent in `build_agent`: gather the tools list and bind them to the model with `create_agent`.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def build_agent():
    model  = ChatOllama(model="llama3.2:1b")
    tools  = ___   # TODO: a list of the two tools (calculator, lookup)
    return create_agent(model, ___)   # TODO: pass the tools list

try:
    agent = build_agent()
    print('agent type   :', type(agent).__name__)
    print('graph nodes  :', set(agent.nodes) - {'__start__'})
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("create_agent returns a runnable CompiledStateGraph", lambda: type(build_agent()).__name__ == "CompiledStateGraph")
expect_true("the agent has a model (decide) node", lambda: "model" in set(build_agent().nodes))
expect_true("the agent has a tools (act) node", lambda: "tools" in set(build_agent().nodes))
expect_true("both tools are bound to the agent", lambda: [t.name for t in [calculator, lookup]] == ["calculator", "lookup"])
expect_true("a single tool binds too", lambda: type(create_agent(ChatOllama(model="llama3.2:1b"), [calculator])).__name__ == "CompiledStateGraph")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the assembled agent on a real question and read the final answer. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        agent = build_agent()
        result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
        print("final:", result["messages"][-1].content)
    else:
        print("No Ollama reachable -- skipping the live run. The agent above is already assembled.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
model + tools -> create_agent = a runnable CompiledStateGraph. It knows its tools and loops model<->tools until it answers; next we run it and read the trace.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>